In [ ]:
import os
import shutil

import networkx as nx
import numpy as np
import scipy.io
import ssgetpy

In [ ]:
os.chdir("../../../")
assert os.path.exists("data")

In [ ]:
matrixes = list(ssgetpy.search(rowbounds=(None, 1000), limit=10000))

cnt = 0
matNames = []
for mat in matrixes:
    # matrix must be a square matrix
    if mat.rows != mat.cols:
        continue

    paths = mat.download(extract=True)
    path = paths[0]
    assert os.path.exists(path + f"/{mat.name}.mtx")
    mtx = scipy.io.mmread(path + f"/{mat.name}.mtx")

    # matrix elements must be non-negative
    minData = np.min(mtx.data)
    if minData < 0:
        continue

    # matrix must be symmetric
    arr = mtx.toarray()
    if not np.allclose(arr, arr.T):
        continue

    # matrix must be connected
    arr[np.diag_indices(arr.shape[0])] = 0
    G = nx.from_numpy_array(arr)
    if not nx.is_connected(G):
        continue

    # pos = nx.kamada_kawai_layout(G)
    # nx.draw(G, pos, with_labels=True)
    # plt.show()

    cnt += 1
    shutil.copy(path + f"/{mat.name}.mtx", f"data/{mat.name}.mtx")
    matNames.append((mat.name, G.number_of_nodes()))

print(f"{cnt=}")

In [ ]:
with open("doc/main/overall/matrixNames.txt", "w") as f:
    for name, _ in matNames:
        assert os.path.exists(f"data/{name}.mtx")
        f.write(name + "\n")

In [ ]:
# make a dict of {matrixName: numNodes}

data = {}
for name, numNodes in matNames:
    data[name] = numNodes

print(data)